# Texture generator: the Python workflow

Generate procedural textures entirely through the importable `texture_generators`
package. This notebook covers discovering materials, Pillow images, NumPy arrays,
all variants, repeatable seeds, material controls, paper measurements and contact
sheets. It finishes by exporting a [Markdown results gallery](results.md) and a
[recipe manifest](manifest.json).

**Python 3.12 or later is required.** From the repository root:

```bash
uv sync --group examples
uv run --group examples jupyter lab examples/texture_generator_workflow.ipynb
```

Select the Python 3 kernel, then **Restart Kernel and Run All Cells**. Alternatively,
run it without a browser:

```bash
uv run --group examples jupyter execute examples/texture_generator_workflow.ipynb
```

For a separate project, install `texture-generator>=0.7.0` and `jupyterlab`, then
open a copy of this notebook. The galvanised examples require package
version 0.7.0 or later. The notebook imports the installed package; no
`sys.path` edits or command-line subprocesses are used for generation.

PNG files live in `images/` beside this notebook. Image previews use Markdown
links, so the notebook contains no embedded image data. A floating-point array
is saved separately in `data/`. Running all cells again overwrites the named
example files, `results.md` and `manifest.json`. Start from a fresh kernel when
regenerating the complete set. Rendering can take a few minutes.

## 1. Import the package and choose an output directory

The distribution is `texture-generator`; its import is `texture_generators`.
Record the environment because exact pixel reproduction also depends on the
package, dependency versions and platform. With the repository lockfile, the
versions below are installed together.

In [1]:
import hashlib
import json
import platform
import sys
from importlib.metadata import version
from pathlib import Path

import numpy as np
from IPython.display import Markdown, display
from PIL import Image

from texture_generators import (
    VARIANTS_BY_MATERIAL,
    all_pairs,
    generate,
    generate_array,
    resolve_variant,
    sample_sheet,
    to_image,
    variants,
)

environment = {
    "python": sys.version.split()[0],
    "platform": sys.platform,
    "architecture": platform.machine(),
    **{name: version(name) for name in ("texture-generator", "numpy", "pillow")},
}
print(json.dumps(environment, indent=2))

# Jupyter normally starts beside the notebook; also support the repository root.
example_dir = Path.cwd()
if (example_dir / "examples" / "texture_generator_workflow.ipynb").is_file():
    example_dir = example_dir / "examples"
image_dir = example_dir / "images"
data_dir = example_dir / "data"
image_dir.mkdir(parents=True, exist_ok=True)
data_dir.mkdir(parents=True, exist_ok=True)
records = []

{
  "python": "3.14.7",
  "platform": "darwin",
  "architecture": "arm64",
  "texture-generator": "0.7.0",
  "numpy": "2.5.2",
  "pillow": "12.3.0"
}


This small notebook helper saves an image, records how it was produced and displays
an external file link. It does not change the generated pixels. The final cell uses
these records to build the complete gallery.

In [2]:
def save_png(
    image: Image.Image,
    filename: str,
    title: str,
    section: str,
    recipe: dict,
) -> None:
    """Save a PNG and register it for the external results gallery."""
    path = image_dir / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    image.save(path, format="PNG")
    relative_path = path.relative_to(example_dir).as_posix()
    records.append(
        {
            "path": relative_path,
            "title": title,
            "section": section,
            "width": image.width,
            "height": image.height,
            "mode": image.mode,
            "recipe": recipe,
            "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
        }
    )
    display(Markdown(f"**{title}**\n\n![{title}]({relative_path})"))

## 2. Discover materials and variants

Use the public registry rather than hard-coding a list for a batch. `variants()`
returns the choices for one material; `all_pairs()` gives every material/variant
combination in registry order.

In [3]:
for material, choices in VARIANTS_BY_MATERIAL.items():
    print(f"{material}: {', '.join(choices)}")
print("Wood choices:", variants("wood"))
pairs = all_pairs()
print(f"{len(pairs)} material/variant combinations")

metal: brushed, radial, polished, heat_tinted, oil_film, anodised_titanium, engine_turned, galvanised
plastic: glossy, matte, textured
wood: board, planks
paper: white, kraft, recycled, newsprint, laid, coated
Wood choices: ['board', 'planks']
19 material/variant combinations


## 3. Generate, save and reopen a Pillow image

An integer size makes a square. A pair means **(width, height)**, and both dimensions
must be at least 2 pixels. `generate()` returns an RGB Pillow image. This is the
README quickstart with an explicit variant and seed.

In [4]:
wood_args = {"size": (640, 480), "seed": 42, "variant": "board"}
wood = generate("wood", **wood_args)
assert wood.mode == "RGB" and wood.size == (640, 480)
save_png(
    wood,
    "wood-board.png",
    "Wood / board — Pillow quickstart",
    "Pillow image",
    {"function": "generate", "material": "wood", **wood_args},
)
with Image.open(image_dir / "wood-board.png") as reopened:
    assert reopened.mode == wood.mode and reopened.size == wood.size
    assert reopened.tobytes() == wood.tobytes()
print("Saved and reopened PNG:", wood.mode, wood.size)

**Wood / board — Pillow quickstart**

![Wood / board — Pillow quickstart](images/wood-board.png)

Saved and reopened PNG: RGB (640, 480)


## 4. Work with floating-point pixels

`generate_array()` returns a NumPy `float32` array shaped **(height, width, 3)**,
with RGB values in `[0, 1]`. Keep the array for numerical work; `to_image()` converts
it to an 8-bit RGB Pillow image for PNG export. It produces the same pixels as
`generate()` called with the same arguments.

The `.npy` file preserves the floating-point values; the PNG is their display
representation. This example checks both round trips.

In [5]:
array_args = {"size": 256, "seed": 7, "variant": "brushed"}
pixels = generate_array("metal", **array_args)
assert pixels.shape == (256, 256, 3) and pixels.dtype == np.float32
assert np.isfinite(pixels).all() and 0 <= pixels.min() <= pixels.max() <= 1
np.save(data_dir / "metal-brushed.npy", pixels)
assert np.array_equal(np.load(data_dir / "metal-brushed.npy"), pixels)
array_image = to_image(pixels)
assert array_image.tobytes() == generate("metal", **array_args).tobytes()
print("Array:", pixels.shape, pixels.dtype)
print("Range:", float(pixels.min()), float(pixels.max()))
save_png(
    array_image,
    "metal-array.png",
    "Metal / brushed — NumPy array as a PNG",
    "NumPy array",
    {
        "function": "generate_array",
        "material": "metal",
        **array_args,
        "conversion": "to_image",
        "array_path": "data/metal-brushed.npy",
    },
)

Array: (256, 256, 3) float32
Range: 0.2375497817993164 0.8513011932373047


**Metal / brushed — NumPy array as a PNG**

![Metal / brushed — NumPy array as a PNG](images/metal-array.png)

## 5. Export every variant in one batch

Each tile below uses an explicit variant, seed **42** and size **384 × 384**.
These are the same files used by the README gallery. Changing the size changes
the render; it is not simply a resized crop of another image.

In [6]:
for material, variant in pairs:
    args = {"size": 384, "seed": 42, "variant": variant}
    tile = generate(material, **args)
    save_png(
        tile,
        f"variants/{material}-{variant}.png",
        f"{material} / {variant}",
        f"All variants — {material}",
        {"function": "generate", "material": material, **args},
    )
print(f"Exported {len(pairs)} variants.")

**metal / brushed**

![metal / brushed](images/variants/metal-brushed.png)

**metal / radial**

![metal / radial](images/variants/metal-radial.png)

**metal / polished**

![metal / polished](images/variants/metal-polished.png)

**metal / heat_tinted**

![metal / heat_tinted](images/variants/metal-heat_tinted.png)

**metal / oil_film**

![metal / oil_film](images/variants/metal-oil_film.png)

...

**paper / recycled**

![paper / recycled](images/variants/paper-recycled.png)

**paper / newsprint**

![paper / newsprint](images/variants/paper-newsprint.png)

**paper / laid**

![paper / laid](images/variants/paper-laid.png)

**paper / coated**

![paper / coated](images/variants/paper-coated.png)

Exported 19 variants.


## 6. Reproduce a render and vary the seed

The same arguments produce identical pixels in the same environment. A different
seed changes the material realisation. Store the material, variant, seed, size
and any overrides together rather than recording only the seed.

In [7]:
first = generate("plastic", size=384, seed=12, variant="matte")
repeat = generate("plastic", size=384, seed=12, variant="matte")
second = generate("plastic", size=384, seed=13, variant="matte")
assert first.tobytes() == repeat.tobytes()
assert first.tobytes() != second.tobytes()
for seed, texture in ((12, first), (13, second)):
    save_png(
        texture,
        f"plastic-seed-{seed}.png",
        f"Plastic / matte — seed {seed}",
        "Reproducible seeds",
        {
            "function": "generate",
            "material": "plastic",
            "size": 384,
            "seed": seed,
            "variant": "matte",
        },
    )
print("Same arguments: identical pixels. Different seed: different pixels.")

**Plastic / matte — seed 12**

![Plastic / matte — seed 12](images/plastic-seed-12.png)

**Plastic / matte — seed 13**

![Plastic / matte — seed 13](images/plastic-seed-13.png)

Same arguments: identical pixels. Different seed: different pixels.


### Let the seed choose the variant

With `variant=None` (or omitted), the first random draw chooses a variant.
`resolve_variant()` tells you which one a concrete seed will choose.
To reproduce this render, **keep the variant omitted**. Passing the resolved name
explicitly skips that first draw and changes the later random draws. A missing
seed uses fresh entropy and cannot predict a separate unseeded render.

In [8]:
automatic_seed = 23
chosen = resolve_variant("metal", seed=automatic_seed)
automatic = generate("metal", size=384, seed=automatic_seed)
assert (
    automatic.tobytes()
    == generate("metal", size=384, seed=automatic_seed, variant=None).tobytes()
)
print("Seed-selected variant:", chosen)
save_png(
    automatic,
    "metal-automatic.png",
    f"Metal / {chosen} — seed-selected variant",
    "Automatic variant selection",
    {
        "function": "generate",
        "material": "metal",
        "size": 384,
        "seed": automatic_seed,
        "variant": None,
        "resolved_variant": chosen,
    },
)

Seed-selected variant: brushed


**Metal / brushed — seed-selected variant**

![Metal / brushed — seed-selected variant](images/metal-automatic.png)

## 7. Control the material

Keyword arguments after `size`, `seed` and `variant` are material-specific.
Here a fixed walnut board is rendered unfinished and oiled, then brushed metal
gets an oil film. The [parameter reference](https://github.com/nmpowell/texture-generator/blob/main/docs/reference.md#material-parameters)
lists supported controls. Unknown extra keywords are currently ignored, so use
those documented names and the appropriate material.

In [9]:
for finish in ("none", "oil"):
    args = {
        "size": 384,
        "seed": 17,
        "variant": "board",
        "species": "walnut",
        "finish": finish,
        "cut": "quartersawn",
        "figure": "plain",
        "mm_across": 180,
        "knots": 0,
        "sapwood": 0,
    }
    controlled = generate("wood", **args)
    save_png(
        controlled,
        f"walnut-{finish}.png",
        f"Walnut / quartersawn — finish {finish}",
        "Material controls",
        {"function": "generate", "material": "wood", **args},
    )
film_args = {
    "size": 384,
    "seed": 17,
    "variant": "brushed",
    "film": {"system": "oil", "nm": (120, 800), "field": "spill"},
}
save_png(
    generate("metal", **film_args),
    "metal-custom-film.png",
    "Brushed metal / oil film",
    "Material controls",
    {"function": "generate", "material": "metal", **film_args},
)

**Walnut / quartersawn — finish none**

![Walnut / quartersawn — finish none](images/walnut-none.png)

**Walnut / quartersawn — finish oil**

![Walnut / quartersawn — finish oil](images/walnut-oil.png)

**Brushed metal / oil film**

![Brushed metal / oil film](images/metal-custom-film.png)

### Every wood species

`SPECIES` in `texture_generators.materials.wood` defines the eight species
below. Omitting `species` chooses one at random per panel. Visible differences
include porosity: pine is a softwood with no vessels; oak and ash are
ring-porous; walnut is semi-ring-porous; and cherry, maple, sapele and mahogany
are diffuse-porous.

Every 384 x 384 tile uses seed 42 and identical arguments apart from `species`:
`variant="board"`, `cut="flatsawn"`, `figure="plain"`, `finish="oil"`,
`colour_variation=0`, `mm_across=180`, `knots=0` and `sapwood=0`. This isolates
each species' anatomy and colour. With `colour_variation=0`, the colour starts
at the species' nominal CIELAB value rather than a board sampled from its
population spread; oil then applies the documented Lab shift. Pinning the cut
follows the species sweep in `src/texture_generators/samples.py`, where cells
differ only by species.

In [10]:
from texture_generators.materials import wood as wood_material

# Keep the pale-to-dark order from samples.py instead of sorting species names.
species_order = [
    "pine",
    "maple",
    "ash",
    "oak",
    "cherry",
    "walnut",
    "sapele",
    "mahogany",
]
assert set(species_order) == set(wood_material.SPECIES), (
    "The wood species list is out of date."
)

for species in species_order:
    args = {
        "size": 384,
        "seed": 42,
        "variant": "board",
        "species": species,
        "cut": "flatsawn",
        "figure": "plain",
        "finish": "oil",
        "colour_variation": 0,
        "mm_across": 180,
        "knots": 0,
        "sapwood": 0,
    }
    tile = generate("wood", **args)
    save_png(
        tile,
        f"species/wood-{species}.png",
        f"Wood / {species}",
        "Wood species",
        {"function": "generate", "material": "wood", **args},
    )
print(f"Exported {len(species_order)} species.")

**Wood / pine**

![Wood / pine](images/species/wood-pine.png)

**Wood / maple**

![Wood / maple](images/species/wood-maple.png)

**Wood / ash**

![Wood / ash](images/species/wood-ash.png)

**Wood / oak**

![Wood / oak](images/species/wood-oak.png)

**Wood / cherry**

![Wood / cherry](images/species/wood-cherry.png)

**Wood / walnut**

![Wood / walnut](images/species/wood-walnut.png)

**Wood / sapele**

![Wood / sapele](images/species/wood-sapele.png)

**Wood / mahogany**

![Wood / mahogany](images/species/wood-mahogany.png)

Exported 8 species.


### Wood chatoyance: move the light

Wood's chatoyance — the way figured grain flashes light and dark — lives
entirely in the reflection: the fibre and ray lobes are what respond to
`light_dir`, and the albedo underneath them does not change at all. The cell
below renders one fixed board six times, rotating only the light's azimuth at
a constant 45 degree elevation, and pastes the tiles into a single strip with
a 6 px white gap between them.

Curly maple shows the effect on the fibre lobe; quartersawn oak's ray fleck
shows it on the crosswise ray lobe mixed into the same lobe by weight
(`ray_tangent`/`ray_weight`). Both boards use `colour_variation=0`, `knots=0`
and `sapwood=0` so the pigment is fixed and only the light direction differs
between tiles.

In [11]:
import math


def light_dir_for(azimuth_deg: float, elevation_deg: float = 45.0) -> tuple:
    """Unit vector towards the light for an azimuth (0 = +x) and elevation, in degrees."""
    az = math.radians(azimuth_deg)
    el = math.radians(elevation_deg)
    return (
        math.cos(az) * math.cos(el),
        math.sin(az) * math.cos(el),
        math.sin(el),
    )


azimuths_deg = [0, 60, 120, 180, 240, 300]
light_dirs = [light_dir_for(az) for az in azimuths_deg]
chatoyance_gap_px = 6
chatoyance_tile_size = 256

chatoyance_boards = [
    (
        "curly-maple",
        "Curly maple",
        {"species": "maple", "figure": "curly", "cut": "flatsawn"},
    ),
    (
        "quartersawn-oak",
        "Quartersawn oak",
        {"species": "oak", "figure": "plain", "cut": "quartersawn"},
    ),
]

for slug, label, species_args in chatoyance_boards:
    shared_args = {
        "size": chatoyance_tile_size,
        "seed": 42,
        "variant": "board",
        "finish": "oil",
        "colour_variation": 0,
        "mm_across": 180,
        "knots": 0,
        "sapwood": 0,
        **species_args,
    }
    tiles = [
        generate("wood", light_dir=light_dir, **shared_args) for light_dir in light_dirs
    ]
    strip = Image.new(
        "RGB",
        (
            chatoyance_tile_size * len(tiles) + chatoyance_gap_px * (len(tiles) - 1),
            chatoyance_tile_size,
        ),
        color="white",
    )
    x = 0
    for tile in tiles:
        strip.paste(tile, (x, 0))
        x += tile.width + chatoyance_gap_px
    save_png(
        strip,
        f"wood-chatoyance-{slug}.png",
        f"Wood / {label} — chatoyance under six light azimuths",
        "Wood chatoyance",
        {
            "function": "generate",
            "material": "wood",
            **shared_args,
            "light_dirs": [list(light_dir) for light_dir in light_dirs],
        },
    )
print(f"Exported {len(chatoyance_boards)} chatoyance strips.")

**Wood / Curly maple — chatoyance under six light azimuths**

![Wood / Curly maple — chatoyance under six light azimuths](images/wood-chatoyance-curly-maple.png)

**Wood / Quartersawn oak — chatoyance under six light azimuths**

![Wood / Quartersawn oak — chatoyance under six light azimuths](images/wood-chatoyance-quartersawn-oak.png)

Exported 2 chatoyance strips.


### Render large, then resample

Wood's relief is carried in real millimetres now, with resolution-independent
normals, so the same board shades the same at any render size — it no longer
has a "native" pixel scale of its own. The documented workflow is still to
render larger than the final size and resample down with a good filter, for
the same reason a photograph is: fine grain streaks and sub-pixel pores are
physical content the renderer resolves at whatever pixel grid it is given,
and a bigger grid resolves more of it, which then anti-aliases into a crisper
small image than rendering directly at that small size ever could.

This is the same relationship the package's own mip test
(`test_a_small_render_matches_a_downsampled_large_one` in
`tests/test_wood_optics.py`) checks between a direct small render and a large
render box-downsampled in linear light. Compare the tile below with the
384 px oak tile in the species gallery above, rendered directly at that size.

In [12]:
oak_species_args = {
    "seed": 42,
    "variant": "board",
    "species": "oak",
    "cut": "flatsawn",
    "figure": "plain",
    "finish": "oil",
    "colour_variation": 0,
    "mm_across": 180,
    "knots": 0,
    "sapwood": 0,
}
large_oak = generate("wood", size=1536, **oak_species_args)
resampled_oak = large_oak.resize((384, 384), Image.LANCZOS)
save_png(
    resampled_oak,
    "wood-oak-1536-to-384.png",
    "Wood / oak — rendered at 1536px, resampled to 384px",
    "Wood scale",
    {
        "function": "generate",
        "material": "wood",
        "size": 1536,
        **oak_species_args,
        "resample": {"to": [384, 384], "filter": "LANCZOS"},
    },
)
print("Rendered at 1536px, resampled to 384px:", resampled_oak.size)

**Wood / oak — rendered at 1536px, resampled to 384px**

![Wood / oak — rendered at 1536px, resampled to 384px](images/wood-oak-1536-to-384.png)

Rendered at 1536px, resampled to 384px: (384, 384)


### Set the brushing direction

`brush_angle` sets the brushing direction in degrees clockwise in image coordinates:
0 is horizontal and 90 is vertical. Finite values are wrapped modulo 360. Set
`material="metal"` and `variant="brushed"` explicitly when using it; leaving
`brush_angle` as `None` or omitting it preserves the previous random direction.

Grooves, scratches, directional reflection, diffraction, sheen and glints follow
the angle; the lighting itself is not rotated. A film may be combined with the
brushed variant. These examples require texture-generator 0.4.0 or later.

Changing the angle can also change fine detail and later seeded choices because
the rotated noise domain may use a different lattice size. Repeat the complete
arguments for identical pixels. Groove diffraction applies to unfilmed metal;
this control does not add diffraction to the film renderer.


In [13]:
for angle in (0, 45, 90, 135):
    args = {
        "size": (384, 256),
        "seed": 42,
        "variant": "brushed",
        "brush_angle": angle,
    }
    brushed = generate("metal", **args)
    save_png(
        brushed,
        f"metal-brush-angle-{angle}.png",
        f"Metal / brushed — brush angle {angle}°",
        "Brushing direction",
        {"function": "generate", "material": "metal", **args},
    )

**Metal / brushed — brush angle 0°**

![Metal / brushed — brush angle 0°](images/metal-brush-angle-0.png)

**Metal / brushed — brush angle 45°**

![Metal / brushed — brush angle 45°](images/metal-brush-angle-45.png)

**Metal / brushed — brush angle 90°**

![Metal / brushed — brush angle 90°](images/metal-brush-angle-90.png)

**Metal / brushed — brush angle 135°**

![Metal / brushed — brush angle 135°](images/metal-brush-angle-135.png)

## 8. Inspect paper measurements

Paper can also fill an `out` dictionary with two scalar fields: `mass` (fibre
coverage) and `formation` (normalised formation). The returned RGB array still
uses the usual contract. This is optional diagnostic data, not a separate set
of PBR texture maps.

Set `creases=0.0` for seamless paper. The field previews below are each scaled
from their own minimum to maximum for display; **their greyscale values are not
physical units**. Use the original arrays for measurements. Ordinary material
outputs are shaded RGB images; the public API does not export general normal,
roughness or albedo maps.

In [14]:
fields = {}
paper_args = {
    "size": 384,
    "seed": 7,
    "variant": "laid",
    "mm_across": 30,
    "creases": 0.0,
}
paper_pixels = generate_array("paper", **paper_args, out=fields)
save_png(
    to_image(paper_pixels),
    "paper-measured.png",
    "Paper / laid — measurement example",
    "Paper measurements",
    {
        "function": "generate_array",
        "material": "paper",
        **paper_args,
        "conversion": "to_image",
        "measurements": ["mass", "formation"],
    },
)
for name in ("mass", "formation"):
    field = fields[name]
    assert field.shape == paper_pixels.shape[:2] and np.isfinite(field).all()
    print(f"{name}: shape={field.shape}, mean={float(field.mean()):.4f}")
    scaled = (field - field.min()) / np.ptp(field)
    preview = to_image(np.repeat(scaled[..., None], 3, axis=2))
    save_png(
        preview,
        f"paper-{name}.png",
        f"Paper / {name} — normalised greyscale preview",
        "Paper measurements",
        {
            "derived_from": "images/paper-measured.png",
            "field": name,
            "conversion": "min-max normalisation, greyscale to RGB",
        },
    )

**Paper / laid — measurement example**

![Paper / laid — measurement example](images/paper-measured.png)

mass: shape=(384, 384), mean=11.2406


**Paper / mass — normalised greyscale preview**

![Paper / mass — normalised greyscale preview](images/paper-mass.png)

formation: shape=(384, 384), mean=-0.0000


**Paper / formation — normalised greyscale preview**

![Paper / formation — normalised greyscale preview](images/paper-formation.png)

## 9. Galvanised surface maps and relighting

These examples show the released procedural visual approximation.
One tile covers **24 × 18 mm**. Height is in micrometres and normals use the
same final height with physical derivative spacing. Fresh zinc F0 stays uniform;
lighting, morphology and angular reflection create the visible pattern.

The following cells save a map contact sheet, a 5 mm scale bar, a light sweep,
a resolution comparison and weathering stages. Recipes and resource hashes are
stored in the gallery manifest. The small resolution images are diagnostics;
they do not certify the full sampling/LOD acceptance gate.


In [15]:
from dataclasses import asdict

from PIL import ImageDraw

from texture_generators import render_material
from texture_generators.materials.galvanised import (
    GalvanisedConfig,
    PreviewConfig,
    build_state,
    sample_state,
)

galv_seed = 42
galv_size = (192, 144)
galv_recipe = GalvanisedConfig(size_mm=(24, 18))
galv_state = build_state(galv_recipe, seed=galv_seed, size=galv_size)
galv_maps = sample_state(galv_state, galv_size)
galv_replay = {
    "function": "build_state/sample_state/render_material",
    "seed": galv_seed,
    "material_key": galv_state.material_key,
    "size": list(galv_size),
    "config": galv_state.config.to_mapping(),
    "generator_version": galv_maps.metadata["generator_version"],
    "resource_hashes": dict(galv_maps.metadata["resource_hashes"]),
    "surface_resource_hashes": dict(galv_maps.metadata["surface_resource_hashes"]),
    "renderer_resource_hashes": dict(galv_maps.metadata["renderer_resource_hashes"]),
    "environment": environment,
    "status": "procedural approximation; measured-material calibration open",
}
galv_image = render_material(galv_maps)
scaled = galv_image.copy()
scale_draw = ImageDraw.Draw(scaled)
scale_pixels = round(5 / galv_state.config.size_mm[0] * galv_size[0])
scale_draw.rectangle(
    (8, galv_size[1] - 24, 18 + scale_pixels, galv_size[1] - 4), fill=(25, 25, 25)
)
scale_draw.line(
    (13, galv_size[1] - 9, 13 + scale_pixels, galv_size[1] - 9), fill="white", width=2
)
scale_draw.text((13, galv_size[1] - 23), "5 mm", fill="white")
save_png(
    scaled,
    "galvanised/regular-scale.png",
    "Galvanised prototype, 5 mm scale bar",
    "Galvanised physical maps",
    {**galv_replay, "scale_bar_mm": 5, "preview": asdict(PreviewConfig())},
)


# Fixed display encodings: neither height nor its normal is normalised per image.
def galv_grey(values):
    return Image.fromarray(np.uint8(np.clip(values, 0, 1) * 255 + 0.5)).convert("RGB")


panels = [
    ("height: -2 to 6 um", galv_grey((galv_maps["height_um"] + 2) / 8)),
    ("signed normal: RGB encode", to_image(galv_maps["normal_ts"] * 0.5 + 0.5)),
    (
        "uniform zinc F0 (sRGB)",
        to_image(
            np.where(
                galv_maps["base_color_linear"] <= 0.0031308,
                12.92 * galv_maps["base_color_linear"],
                1.055 * galv_maps["base_color_linear"] ** (1 / 2.4) - 0.055,
            )
        ),
    ),
    ("roughness: 0 to 1", galv_grey(galv_maps["roughness"])),
    ("anisotropy: 0 to 1", galv_grey(galv_maps["anisotropy"])),
    ("zinc coverage: 0 to 1", galv_grey(galv_maps["metallic"])),
]
contact = Image.new("RGB", (galv_size[0] * 3, (galv_size[1] + 22) * 2), (25, 25, 25))
label = ImageDraw.Draw(contact)
for index, (title, panel) in enumerate(panels):
    x = (index % 3) * galv_size[0]
    y = (index // 3) * (galv_size[1] + 22)
    contact.paste(panel, (x, y + 22))
    label.text((x + 4, y + 4), title, fill="white")
save_png(
    contact,
    "galvanised/maps.png",
    "Galvanised map diagnostics with fixed display ranges",
    "Galvanised physical maps",
    {**galv_replay, "display_panels": [title for title, _ in panels]},
)

**Galvanised prototype, 5 mm scale bar**

![Galvanised prototype, 5 mm scale bar](images/galvanised/regular-scale.png)

**Galvanised map diagnostics with fixed display ranges**

![Galvanised map diagnostics with fixed display ranges](images/galvanised/maps.png)

### Four galvanised presets

The regular, minimised, weathered and wet-storage presets use the same seed,
physical area, resolution and default light. The comparison shows the intended
changes to grain scale and surface condition without changing the light.


In [16]:
galv_presets = ("regular", "minimised", "weathered", "wet_storage")
galv_preset_size = (192, 144)
galv_preset_mm = (24, 18)
galv_preset_tiles = []
for preset in galv_presets:
    recipe = GalvanisedConfig(preset=preset, size_mm=galv_preset_mm)
    tile = generate(
        "metal",
        size=galv_preset_size,
        seed=galv_seed,
        variant="galvanised",
        galvanised=recipe,
    )
    galv_preset_tiles.append(tile)
    save_png(
        tile,
        f"galvanised/preset-{preset}.png",
        f"Galvanised / {preset} preset",
        "Galvanised presets",
        {
            "function": "generate",
            "material": "metal",
            "variant": "galvanised",
            "size": list(galv_preset_size),
            "seed": galv_seed,
            "galvanised": recipe.to_mapping(),
        },
    )

preset_gap = 8
preset_label_height = 24
preset_comparison = Image.new(
    "RGB",
    (
        2 * galv_preset_size[0] + 3 * preset_gap,
        2 * (galv_preset_size[1] + preset_label_height) + 3 * preset_gap,
    ),
    color=(245, 245, 245),
)
preset_draw = ImageDraw.Draw(preset_comparison)
for index, (preset, tile) in enumerate(
    zip(galv_presets, galv_preset_tiles, strict=True)
):
    x = preset_gap + (index % 2) * (galv_preset_size[0] + preset_gap)
    y = preset_gap + (index // 2) * (
        galv_preset_size[1] + preset_label_height + preset_gap
    )
    preset_draw.text((x, y), preset.replace("_", " ").title(), fill=(20, 20, 20))
    preset_comparison.paste(tile, (x, y + preset_label_height))
save_png(
    preset_comparison,
    "galvanised/preset-comparison.png",
    "Galvanised — four presets, one seed and light",
    "Galvanised presets",
    {
        "function": "generate",
        "material": "metal",
        "variant": "galvanised",
        "presets": list(galv_presets),
        "size": list(galv_preset_size),
        "size_mm": list(galv_preset_mm),
        "seed": galv_seed,
    },
)

**Galvanised / regular preset**

![Galvanised / regular preset](images/galvanised/preset-regular.png)

**Galvanised / minimised preset**

![Galvanised / minimised preset](images/galvanised/preset-minimised.png)

**Galvanised / weathered preset**

![Galvanised / weathered preset](images/galvanised/preset-weathered.png)

**Galvanised / wet_storage preset**

![Galvanised / wet_storage preset](images/galvanised/preset-wet_storage.png)

**Galvanised — four presets, one seed and light**

![Galvanised — four presets, one seed and light](images/galvanised/preset-comparison.png)

### Change the light while retaining the same maps

The same `MaterialMaps` object is reused. No new grain, weather or feature seed
is drawn for a lighting change. Display exposure remains at its physical default.


In [17]:
for azimuth in (15, 105):
    preview = PreviewConfig(rig="oblique", light_azimuth_deg=azimuth)
    save_png(
        render_material(galv_maps, preview=preview),
        f"galvanised/light-{azimuth}.png",
        f"Same galvanised surface, light azimuth {azimuth} degrees",
        "Galvanised relighting",
        {**galv_replay, "preview": asdict(preview)},
    )

**Same galvanised surface, light azimuth 15 degrees**

![Same galvanised surface, light azimuth 15 degrees](images/galvanised/light-15.png)

**Same galvanised surface, light azimuth 105 degrees**

![Same galvanised surface, light azimuth 105 degrees](images/galvanised/light-105.png)

### Resample the same physical tile

Pixel density changes; the 24 × 18 mm extent, material key and physical feature
records are fixed. The saved files retain their native pixel sizes.


In [18]:
for resolution in ((48, 36), (96, 72), (192, 144)):
    sampled = (
        galv_maps if resolution == galv_size else sample_state(galv_state, resolution)
    )
    save_png(
        render_material(sampled),
        f"galvanised/resolution-{resolution[0]}.png",
        f"Same 24 x 18 mm tile at {resolution[0]} x {resolution[1]} pixels",
        "Galvanised resolution diagnostics",
        {**galv_replay, "size": list(resolution), "preview": asdict(PreviewConfig())},
    )

**Same 24 x 18 mm tile at 48 x 36 pixels**

![Same 24 x 18 mm tile at 48 x 36 pixels](images/galvanised/resolution-48.png)

**Same 24 x 18 mm tile at 96 x 72 pixels**

![Same 24 x 18 mm tile at 96 x 72 pixels](images/galvanised/resolution-96.png)

**Same 24 x 18 mm tile at 192 x 144 pixels**

![Same 24 x 18 mm tile at 192 x 144 pixels](images/galvanised/resolution-192.png)

### Age a fixed substrate

Exposure is a normalised authoring coordinate, not elapsed days. The environment
and weather seed stay fixed while exposure increases. Deposits change geometry
and visible dielectric coverage; exposed zinc retains its intrinsic optics.


In [19]:
for exposure in (0.0, 0.5, 1.0):
    weather_recipe = GalvanisedConfig(
        size_mm=(24, 18),
        exposure=exposure,
        wetness=0.9,
        confinement=0.85,
        salt_exposure=0.2,
        white_stain=0.85,
    )
    weather_state = build_state(weather_recipe, seed=galv_seed, size=galv_size)
    assert np.array_equal(
        weather_state.partition.points_mm, galv_state.partition.points_mm
    )
    weather_maps = sample_state(weather_state, galv_size)
    save_png(
        render_material(weather_maps),
        f"galvanised/exposure-{exposure:.1f}.png",
        f"Fixed galvanised substrate, exposure {exposure:.1f}",
        "Galvanised weathering",
        {
            **galv_replay,
            "config": weather_state.config.to_mapping(),
            "preview": asdict(PreviewConfig()),
        },
    )

**Fixed galvanised substrate, exposure 0.0**

![Fixed galvanised substrate, exposure 0.0](images/galvanised/exposure-0.0.png)

**Fixed galvanised substrate, exposure 0.5**

![Fixed galvanised substrate, exposure 0.5](images/galvanised/exposure-0.5.png)

**Fixed galvanised substrate, exposure 1.0**

![Fixed galvanised substrate, exposure 1.0](images/galvanised/exposure-1.0.png)

## 10. Make a labelled contact sheet

`sample_sheet()` includes every material/variant pair in a single Pillow image.
`size` is the size of each tile; `columns` controls the layout. The master seed
draws a separate seed for each tile, so these tiles differ from the seed-42
batch above. Labels abbreviate tile seeds; keep the master seed and call arguments
to regenerate the whole sheet.

In [20]:
sheet_args = {"size": 192, "seed": 7, "columns": 4}
sheet = sample_sheet(**sheet_args)
save_png(
    sheet,
    "contact-sheet.png",
    "All materials — labelled contact sheet",
    "Contact sheet",
    {"function": "sample_sheet", **sheet_args},
)
print("Contact sheet:", sheet.mode, sheet.size)

**All materials — labelled contact sheet**

![All materials — labelled contact sheet](images/contact-sheet.png)

Contact sheet: RGB (798, 1066)


## 11. Handle invalid input

Unknown materials, unknown variants and dimensions below 2 pixels raise
`ValueError`. Validate application input or handle the error at your boundary.

In [21]:
try:
    generate("wood", size=1, seed=42, variant="board")
except ValueError as error:
    print(f"Expected validation error: {error}")

Expected validation error: size must be at least 2x2, got 1x1


## 12. Write the complete results gallery

The Markdown document references every PNG saved in this run. The JSON manifest
records dimensions, recipes, environment versions and PNG checksums. Recipe
entries include descriptive fields such as `function` and `conversion`; pass
only the documented generator arguments back to the API.

Run all cells when changing parameters so the PNGs, notebook outputs, gallery
and manifest agree. Old files from recipes you remove are not automatically deleted.

In [22]:
paths = [record["path"] for record in records]
assert len(paths) == len(set(paths)), "Use a different filename for each result."
manifest = {"environment": environment, "images": records}
(example_dir / "manifest.json").write_text(
    json.dumps(manifest, indent=2) + "\n", encoding="utf-8"
)
lines = [
    "# Texture generator: notebook results",
    "",
    "Generated by [the Python workflow notebook](texture_generator_workflow.ipynb).",
    "Run all cells to regenerate these PNG files and this document. Images are",
    "stored separately in `images/`; no image data is embedded in the notebook.",
    "",
    "Every image from the notebook is shown below. The [recipe manifest](manifest.json)",
    "records the exact arguments and PNG checksums. Exact reproduction also depends",
    "on the dependency versions and platform. The saved floating-point array is",
    "[metal-brushed.npy](data/metal-brushed.npy).",
    "",
    "## Environment",
    "",
    "```json",
    json.dumps(environment, indent=2),
    "```",
    "",
    "All variant-gallery tiles use seed 42 and size 384 x 384. The contact sheet",
    "uses its own master seed and draws individual tile seeds.",
    "",
    "Paper measurement previews use min-max greyscale normalisation for display;",
    "their displayed brightness is not a physical unit.",
    "",
]
previous_section = None
for record in records:
    if record["section"] != previous_section:
        lines.extend([f"## {record['section']}", ""])
        previous_section = record["section"]
    lines.extend(
        [
            f"### {record['title']}",
            "",
            f"{record['width']} x {record['height']} · {record['mode']}",
            "",
            f"![{record['title']}]({record['path']})",
            "",
        ]
    )
(example_dir / "results.md").write_text("\n".join(lines), encoding="utf-8")
print(f"Saved {len(records)} PNGs, the NumPy array, manifest.json and results.md.")
display(Markdown("[Open the complete results gallery](results.md)"))

Saved 61 PNGs, the NumPy array, manifest.json and results.md.


[Open the complete results gallery](results.md)